<a href="https://colab.research.google.com/github/DaniloDuque/logistic-regression/blob/main/src/tp2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP2 — Regresión Logística

## Setup

In [ ]:
import os, sys

if 'google.colab' in sys.modules:
    if not os.path.exists('logistic-regression'):
        os.system('git clone https://github.com/DaniloDuque/logistic-regression.git')
    os.chdir('logistic-regression')

sys.path.insert(0, os.path.abspath('src'))

---
# Sección 1 — Algoritmo de Regresión Logística

## 1.a — Usar el código del perceptrón como base
DOCUMENTACION EN LATEX: Explique en el reporte cada una de las modificaciones necesarias al código del perceptrón, utilizando como referencia las diferencias entre ambos algoritmos.

### Pruebas unitarias
2 pruebas unitarias (con su objetivo, entradas, salidas esperadas y resultados) para las funciones modificadas en el algoritmo del perceptron base.

---
## 1.b — Experimentos: Datos separables vs no separables

### Imports

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from data_generator import generate_data
from logistic_regression import LogisticRegression
from trainer import train_with_history, compute_mae
from metrics import run_experiment, print_mae_table, print_runs_table
from visualization import plot_results

STEPS = 1000
ALPHA = 0.1

### Experiment 1 — Linearly separable data

Two classes generated with `cluster_std=1.0` (compact, well-separated clusters).
The model is expected to converge quickly and achieve a low MAE.

In [ ]:
X_train_s, X_test_s, y_train_s, y_test_s = generate_data(separable=True, n_samples=500, random_state=42)

model_s = LogisticRegression(torch.zeros(X_train_s.shape[1], 1))
errors_s = train_with_history(model_s, X_train_s, y_train_s, steps=STEPS, alpha=ALPHA)

mae_train_s = compute_mae(y_train_s, model_s.forward(X_train_s))
mae_test_s  = compute_mae(y_test_s,  model_s.forward(X_test_s))

print(f"Iterations: {STEPS}")
print_mae_table('Linearly separable', mae_train_s, mae_test_s)

plot_results(model_s, X_train_s, y_train_s, errors_s, 'Experiment 1 — Linearly separable data')

### Experiment 2 — Non-linearly separable data

Two classes generated with `cluster_std=4.0` (overlapping clusters).
The model is expected to struggle and achieve a higher MAE.

In [ ]:
X_train_ns, X_test_ns, y_train_ns, y_test_ns = generate_data(separable=False, n_samples=500, random_state=42)

model_ns = LogisticRegression(torch.zeros(X_train_ns.shape[1], 1))
errors_ns = train_with_history(model_ns, X_train_ns, y_train_ns, steps=STEPS, alpha=ALPHA)

mae_train_ns = compute_mae(y_train_ns, model_ns.forward(X_train_ns))
mae_test_ns  = compute_mae(y_test_ns,  model_ns.forward(X_test_ns))

print(f"Iterations: {STEPS}")
print_mae_table('Non-linearly separable', mae_train_ns, mae_test_ns)

plot_results(model_ns, X_train_ns, y_train_ns, errors_ns, 'Experiment 2 — Non-linearly separable data')

### Comparative MAE table

In [ ]:
print(f"{'Case':<30} {'MAE Train':>10} {'MAE Test':>10}")
print("-" * 55)
print(f"{'Linearly separable':<30} {mae_train_s:>10.4f} {mae_test_s:>10.4f}")
print(f"{'Non-linearly separable':<30} {mae_train_ns:>10.4f} {mae_test_ns:>10.4f}")

### 10 runs — Mean MAE and standard deviation

In [ ]:
maes_s  = run_experiment(separable=True,  steps=STEPS, alpha=ALPHA)
maes_ns = run_experiment(separable=False, steps=STEPS, alpha=ALPHA)

print_runs_table(maes_s, maes_ns)

---
# Sección 2 — Regresión Logística y LLMs para clasificación de textos

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="saul1917/FEINA",
    filename="FEINA_1.xlsx",
    repo_type="dataset"
)

df = pd.read_excel(file_path)
print(df.shape)
print(df.head())
print(df.columns.tolist())

### BERT embeddings — sanity check

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
import numpy as np

# ── Model 1: Spanish BERT (dccuchile) ─────────────────────────────
tokenizer1 = AutoTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
model1 = AutoModelForMaskedLM.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
model1.eval()

# ── Model 2: BERTIN - Spanish RoBERTa ─────────────────────────────
tokenizer2 = AutoTokenizer.from_pretrained("bertin-project/bertin-roberta-base-spanish")
model2 = AutoModelForMaskedLM.from_pretrained("bertin-project/bertin-roberta-base-spanish")
model2.eval()

# ── Embedding via CLS token ────────────────────────────────────────
def get_embedding(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt",
                       truncation=True, max_length=512, padding=True)
    with torch.no_grad():
        outputs = model.base_model(**inputs)
    return outputs.last_hidden_state[:, 0, :].squeeze().numpy()

# ── Sanity check ───────────────────────────────────────────────────
test = "El índice de capitalización bursátil refleja la valoración agregada."
emb1 = get_embedding(test, model1, tokenizer1)
emb2 = get_embedding(test, model2, tokenizer2)

print(f"Model 1 (BERT-es) shape:   {emb1.shape}")
print(f"Model 2 (BERTIN) shape:    {emb2.shape}")